# NLP with Transformers — Complete Reference Notebook

**Topics Covered:**

| § | Topic | Key Concept |
|---|---|---|
| 1 | Tokenization | Text → integer token IDs |
| 2 | Embeddings | IDs → dense vectors (token + positional + segment) |
| 3 | Self-Attention | Every token attends to every other token |
| 4 | Multi-Head Attention | Parallel attention heads in subspaces |
| 5 | Transformer Encoder | Bidirectional deep context (BERT-style) |
| 6 | Transformer Decoder | Auto-regressive generation (GPT-style) |
| 7 | Pre-training Tasks | MLM, NSP, CLM — self-supervised learning |
| 8 | Fine-tuning | Classification, NER, QA — task adaptation |
| 9 | Seq2Seq & Decoding | Translation/Summarization + Beam Search |
| 10 | Evaluation Metrics | BLEU, ROUGE, F1, seqeval |

---
**Requirements:**
```
pip install torch transformers datasets evaluate sacrebleu rouge_score
```


## Setup & Imports

In [ ]:
# Install required packages (run once)
# !pip install torch transformers datasets evaluate sacrebleu rouge_score

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    DataCollatorWithPadding,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    pipeline,
)
from datasets import load_dataset
import evaluate

print("All imports successful!")
print(f"PyTorch version  : {torch.__version__}")
import transformers
print(f"Transformers ver : {transformers.__version__}")


---
## §1 — Tokenization

### What is it?
Tokenization converts raw text into discrete integer IDs that a model can process.
Modern transformers use **sub-word** schemes:
- **WordPiece** — used by BERT; greedily matches longest known sub-word
- **BPE (Byte-Pair Encoding)** — used by GPT; iteratively merges most frequent pairs
- **SentencePiece** — used by T5/mT5; language-agnostic, works on raw bytes

### Why do we need it?
- Raw characters → sequences too long, no word-level meaning
- Full words → can't handle rare/misspelled/compound words (OOV problem)
- Sub-words → balance: covers any word, manageable sequence length

### How it works
1. Build vocabulary of most frequent sub-words (e.g., 30,522 tokens for BERT)
2. Split words into the longest known sub-word pieces
3. Map each piece to a unique integer ID
4. Add special tokens: `[CLS]` (classification), `[SEP]` (separator), `[PAD]` (padding)

### Key outputs
| Output | Shape | Purpose |
|--------|-------|---------|
| `input_ids` | `[batch, seq_len]` | Integer token IDs |
| `attention_mask` | `[batch, seq_len]` | 1 = real token, 0 = padding |
| `token_type_ids` | `[batch, seq_len]` | 0 = sentence A, 1 = sentence B |


In [ ]:
# ── 1a: Basic Tokenization ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Transformers revolutionized NLP by enabling parallelizable training."

# Encode with padding and truncation
encoding = tokenizer(
    text,
    return_tensors="pt",   # PyTorch tensors
    padding=True,          # pad to longest in batch
    truncation=True,       # cut sequences longer than max_length
    max_length=128,
)

print("── Basic Encoding ──────────────────────────────────────────────")
print(f"Input IDs shape  : {encoding['input_ids'].shape}")   # [1, seq_len]
print(f"Input IDs        : {encoding['input_ids']}")
print(f"Attention mask   : {encoding['attention_mask']}")

# Decode back to human-readable tokens
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
print(f"\nTokens           : {tokens}")
print(f"  → Note '##' prefix = continuation sub-word (still part of previous word)")


In [ ]:
# ── 1b: Batch Encoding ───────────────────────────────────────────────────────
batch = tokenizer(
    ["Hello world!", "Attention is all you need."],
    return_tensors="pt",
    padding=True,     # shorter sentence gets padded to match longer
    truncation=True,
)

print("── Batch Encoding ──────────────────────────────────────────────")
print(f"input_ids shape  : {batch['input_ids'].shape}")  # [2, max_seq_len]
print(f"Batch input IDs  :\n{batch['input_ids']}")
print(f"Batch attn mask  :\n{batch['attention_mask']}")
print("  → Row 0 is padded at end to match length of row 1")


In [ ]:
# ── 1c: Sentence-Pair Encoding (for QA / NSP tasks) ─────────────────────────
# BERT's NSP task and QA tasks encode two sentences as one sequence:
# [CLS] sentence_A [SEP] sentence_B [SEP]
# token_type_ids: 0 for sentence A, 1 for sentence B

pair = tokenizer(
    "What is NLP?",
    "Natural Language Processing is a field of AI.",
    return_tensors="pt",
)

print("── Sentence Pair Encoding ─────────────────────────────────────")
tokens = tokenizer.convert_ids_to_tokens(pair["input_ids"][0])
print(f"Tokens           : {tokens}")
print(f"Token type IDs   : {pair['token_type_ids']}")
print("  → 0 = Question tokens (Sentence A)")
print("  → 1 = Context tokens (Sentence B)")


In [ ]:
# ── 1d: Compare Tokenizers (BPE vs WordPiece) ────────────────────────────────
from transformers import GPT2Tokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")

test_word = "unhappiness"

bert_tokens = bert_tok.tokenize(test_word)
gpt2_tokens = gpt2_tok.tokenize(test_word)

print("── Tokenizer Comparison ───────────────────────────────────────")
print(f"Word             : '{test_word}'")
print(f"BERT (WordPiece) : {bert_tokens}")
print(f"  → Splits into sub-words: prefix '##' marks continuation")
print(f"GPT-2 (BPE)      : {gpt2_tokens}")
print(f"  → Byte-level BPE, space prefix 'Ġ' marks word start")


---
## §2 — Embeddings

### What are they?
An embedding maps a discrete token ID to a **continuous dense vector**.
Transformers sum **three** embedding components per token:

| Component | What it encodes | Shape |
|-----------|----------------|-------|
| Token embedding | Meaning of the token itself | `[vocab_size, hidden_size]` |
| Positional encoding | Where in the sequence the token appears | `[max_pos, hidden_size]` |
| Segment embedding | Which sentence (A or B) the token belongs to | `[2, hidden_size]` |

### Why do we need embeddings?
- Neural networks need floating-point inputs, not integers
- Dense vectors have **geometry**: similar meanings → nearby vectors
- Positional encoding is essential because self-attention is **permutation invariant** — without it, "dog bites man" and "man bites dog" look identical

### Two positional encoding flavours
- **Learned** (BERT): trainable lookup table, flexible, limited to max_position seen during training
- **Sinusoidal** (original Transformer): `PE(pos,2i) = sin(pos/10000^(2i/d))`, fixed, can extrapolate to longer sequences


In [ ]:
# ── 2a: TransformerEmbeddings — built from scratch ─────────────────────────
class TransformerEmbeddings(nn.Module):
    """
    Combines token + positional + segment embeddings.
    This is exactly what BERT's BertEmbeddings class does internally.
    """
    def __init__(self, vocab_size, hidden_size, max_position=512,
                 num_segments=2, dropout=0.1):
        super().__init__()

        # (a) Token embedding: lookup table [vocab_size × hidden_size]
        #     padding_idx=0 → the [PAD] token's embedding is kept at zero,
        #     and gradients through it are zeroed (no update from padding)
        self.token_embed = nn.Embedding(vocab_size, hidden_size, padding_idx=0)

        # (b) Positional embedding: learned [max_position × hidden_size]
        #     Tells the model WHERE each token is in the sequence
        self.position_embed = nn.Embedding(max_position, hidden_size)

        # (c) Segment embedding: [2 × hidden_size]
        #     0 = Sentence A, 1 = Sentence B
        self.segment_embed = nn.Embedding(num_segments, hidden_size)

        # LayerNorm: normalizes each sample's activations to mean≈0, std≈1
        # eps=1e-12 prevents division-by-zero for near-zero variance inputs
        self.layer_norm = nn.LayerNorm(hidden_size, eps=1e-12)

        # Dropout: randomly zeroes out neurons during training → regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, token_type_ids=None):
        seq_len = input_ids.size(1)

        # Build position indices [0, 1, 2, ..., seq_len-1] for each sample
        positions = torch.arange(seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand_as(input_ids)

        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)

        # Sum all three embeddings (element-wise addition in hidden_size space)
        embeddings = (
            self.token_embed(input_ids)           # semantic: what is this token?
            + self.position_embed(positions)       # positional: where is it?
            + self.segment_embed(token_type_ids)   # structural: which sentence?
        )

        # Normalize then regularize
        return self.dropout(self.layer_norm(embeddings))


# ── Demonstration ──────────────────────────────────────────────────────────
embed = TransformerEmbeddings(vocab_size=30522, hidden_size=768)
dummy_ids = torch.randint(0, 30522, (2, 16))  # batch=2, seq_len=16
output = embed(dummy_ids)

print("── TransformerEmbeddings Demo ─────────────────────────────────")
print(f"Input shape      : {dummy_ids.shape}       (batch=2, seq_len=16)")
print(f"Output shape     : {output.shape}   (batch=2, seq_len=16, hidden=768)")
print(f"Token embed size : {embed.token_embed.weight.shape}")
print(f"Pos embed size   : {embed.position_embed.weight.shape}")
print(f"Seg embed size   : {embed.segment_embed.weight.shape}")
print(f"\nSample output (first token, first 8 dims):")
print(f"  {output[0, 0, :8].detach().numpy().round(3)}")


In [ ]:
# ── 2b: Sinusoidal Positional Encoding (original Transformer) ───────────────
import matplotlib.pyplot as plt

class SinusoidalPositionalEncoding(nn.Module):
    """
    Fixed (non-learned) encoding from 'Attention Is All You Need' (Vaswani 2017).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    WHY: different frequencies per dimension → each position has a unique fingerprint.
    BENEFIT: can extrapolate to longer sequences than seen during training.
    """
    def __init__(self, hidden_size, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, hidden_size)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        # Exponentially spaced frequencies
        div_term = torch.exp(
            torch.arange(0, hidden_size, 2).float()
            * (-math.log(10000.0) / hidden_size)
        )
        pe[:, 0::2] = torch.sin(position * div_term)   # even dims → sine
        pe[:, 1::2] = torch.cos(position * div_term)   # odd dims  → cosine
        self.register_buffer("pe", pe.unsqueeze(0))    # [1, max_len, hidden]

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]              # add to token embeddings


# Visualise the encoding patterns
pe_module = SinusoidalPositionalEncoding(hidden_size=64, max_len=50)
pe_matrix = pe_module.pe[0].numpy()  # [50, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of encoding values
im = axes[0].imshow(pe_matrix[:30, :32], aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Position in sequence')
axes[0].set_title('Sinusoidal positional encoding\n(first 30 positions × 32 dims)')
plt.colorbar(im, ax=axes[0])

# Show a few individual dimensions
for dim in [0, 4, 8, 16]:
    axes[1].plot(pe_matrix[:, dim], label=f'dim {dim}', linewidth=1.5)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Encoding value')
axes[1].set_title('Individual dimensions across positions\n(different frequencies)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/positional_encoding.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nNote: Lower dimensions oscillate FAST (high frequency → fine position detail)")
print("      Higher dimensions oscillate SLOW (low frequency → coarse position info)")


---
## §3 — Scaled Dot-Product Self-Attention

### What is it?
Self-attention allows every token to **look at every other token** and compute a context-aware representation by taking a weighted sum of all value vectors.

### The Q, K, V framework
Each token projects into three roles simultaneously:
- **Query (Q)** — "What am I looking for?"
- **Key (K)** — "What do I contain / advertise?"
- **Value (V)** — "What information do I carry?"

### The formula
$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Why the √d_k scaling?
In high dimensions, dot products grow large → softmax saturates → near-zero gradients → no learning.
Dividing by √d_k keeps the variance of dot products at ~1 regardless of dimension size.

### Why self-attention beats RNNs?
| Property | RNN | Self-Attention |
|---|---|---|
| Long-range dependencies | Poor (gradient vanishing) | Perfect (direct connection) |
| Parallelism | Sequential (slow) | Fully parallel (fast) |
| Path length between positions | O(n) | O(1) |


In [ ]:
# ── 3a: Scaled Dot-Product Attention ────────────────────────────────────────
class ScaledDotProductAttention(nn.Module):
    """
    Core attention operation.
    Input shapes: Q, K, V = [batch, heads, seq_len, d_k]
    Output shape: [batch, heads, seq_len, d_k], attention_weights
    """
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)  # dimension per head

        # Step 1: Compute raw attention scores (dot product similarity)
        # [batch, heads, seq_q, d_k] × [batch, heads, d_k, seq_k]
        # → [batch, heads, seq_q, seq_k]
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # Step 2: Scale to prevent softmax saturation
        scores = scores / math.sqrt(d_k)

        # Step 3: Apply mask (padding positions → -∞ → softmax weight ≈ 0)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Step 4: Softmax → probability distribution over all positions
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Step 5: Weighted sum of values
        output = torch.matmul(attn_weights, V)
        # [batch, heads, seq_q, seq_k] × [batch, heads, seq_k, d_k]
        # → [batch, heads, seq_q, d_k]
        return output, attn_weights


# ── Demo: Visualise attention weights ────────────────────────────────────────
attn = ScaledDotProductAttention(dropout=0.0)

# Small example: batch=1, heads=1, seq=5, d_k=8
seq_len, d_k = 5, 8
Q = torch.randn(1, 1, seq_len, d_k)
K = torch.randn(1, 1, seq_len, d_k)
V = torch.randn(1, 1, seq_len, d_k)

output, weights = attn(Q, K, V)
print("── Scaled Dot-Product Attention ──────────────────────────────")
print(f"Q, K, V shape    : {Q.shape}")
print(f"Output shape     : {output.shape}")
print(f"Attention weights:\n{weights[0, 0].detach().numpy().round(3)}")
print(f"  → Each row sums to 1.0 (probability distribution over positions)")
print(f"  → Row i = how much position i attends to each position j")


In [ ]:
# ── 3b: Visualise Attention on a Real Sentence ──────────────────────────────
import matplotlib.pyplot as plt

# Use a pre-trained BERT to extract real attention weights
from transformers import BertModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
model.eval()

sentence = "The bank approved the loan near the river bank."
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = model(**inputs)

# outputs.attentions: tuple of 12 tensors, one per layer
# Each tensor: [batch, num_heads, seq_len, seq_len]
layer = 10          # higher layers = more semantic
head  = 5           # different heads capture different patterns
attn_matrix = outputs.attentions[layer][0, head].numpy()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(attn_matrix, cmap="Blues", vmin=0, vmax=attn_matrix.max())

ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(tokens, fontsize=9)
ax.set_title(f"BERT attention weights — Layer {layer+1}, Head {head+1}\n"
             f"Row = query token, Column = key token, Brightness = attention weight",
             fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('/home/claude/attention_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nNote: Look for 'bank' row — it should attend strongly to 'loan'/'approved'")
print("      (financial context) vs 'river'/'near' (geographic context)")


---
## §4 — Multi-Head Attention

### What is it?
Multi-head attention runs **H independent attention operations in parallel**, each in a lower-dimensional subspace (d_k = d_model / H), then concatenates and projects back to d_model.

### Why multiple heads?
A single attention head can only compute one kind of relationship at a time. Multiple heads allow the model to **simultaneously attend to different aspects**:
- Head 1 might focus on syntactic dependencies
- Head 2 might focus on coreference (pronoun → noun)
- Head 3 might focus on positional proximity
- etc.

### The math
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_H) W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

BERT-base: 12 heads × 64-dim each = 768-dim total


In [ ]:
# ── 4a: Multi-Head Attention — full implementation ──────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        assert hidden_size % num_heads == 0, \
            f"hidden_size ({hidden_size}) must be divisible by num_heads ({num_heads})"

        self.num_heads = num_heads
        self.d_k = hidden_size // num_heads   # dimension per head

        # Four linear projections: Q, K, V, and output
        # We project the full hidden_size at once (all heads combined)
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)
        self.W_o = nn.Linear(hidden_size, hidden_size)

        self.attn  = ScaledDotProductAttention(dropout)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x, batch_size):
        """
        Reshape [batch, seq, hidden] → [batch, heads, seq, d_k]
        This splits the hidden dimension into num_heads subspaces.
        """
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)   # [batch, heads, seq, d_k]

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1. Project all heads at once, then split into subspaces
        Q = self.split_heads(self.W_q(query), batch_size)
        K = self.split_heads(self.W_k(key),   batch_size)
        V = self.split_heads(self.W_v(value),  batch_size)

        # 2. Run attention in each head's subspace (parallel)
        attn_out, attn_weights = self.attn(Q, K, V, mask)

        # 3. Concatenate heads: [batch, heads, seq, d_k] → [batch, seq, hidden]
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch_size, -1, self.num_heads * self.d_k)

        # 4. Final projection (mix information across heads)
        return self.W_o(attn_out), attn_weights


# ── Demo ──────────────────────────────────────────────────────────────────────
mha = MultiHeadAttention(hidden_size=768, num_heads=12)
x   = torch.randn(2, 16, 768)        # batch=2, seq=16, hidden=768
out, weights = mha(x, x, x)          # self-attention: query=key=value=x

print("── Multi-Head Attention ──────────────────────────────────────")
print(f"Input shape      : {x.shape}")
print(f"Output shape     : {out.shape}")      # same as input
print(f"Weight shape     : {weights.shape}")  # [2, 12 heads, 16, 16]
print(f"d_k per head     : {mha.d_k}  (768 / 12 heads)")
print(f"\nTotal parameters in MHA:")
total = sum(p.numel() for p in mha.parameters())
print(f"  {total:,}  ({total/1e6:.2f}M)")


---
## §5 — Transformer Encoder (BERT-style)

### What is it?
A stack of N identical encoder layers. Each layer has two sublayers:
1. **Multi-head self-attention** (all tokens attend to all tokens, bidirectionally)
2. **Position-wise Feed-Forward Network** (FFN): two linear layers with GELU, expanding to 4× hidden_size

Both sublayers use **Add & Norm** (residual connection + LayerNorm).

### Why bidirectional?
Understanding requires full context: "bank" means different things in "river bank" vs "bank account".  
The encoder can see ALL tokens simultaneously → rich contextual representations.

### Residual connections
`output = LayerNorm(x + sublayer(x))`  
Gradients can flow DIRECTLY to earlier layers, preventing vanishing gradients in deep networks (12-24 layers).

### BERT-base vs BERT-large
| | BERT-base | BERT-large |
|---|---|---|
| Layers | 12 | 24 |
| Hidden size | 768 | 1024 |
| Attention heads | 12 | 16 |
| Parameters | 110M | 340M |


In [ ]:
# ── 5a: Encoder Layer ────────────────────────────────────────────────────────
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    Architecture: Linear → GELU → Dropout → Linear
    Expansion: hidden_size → 4×hidden_size → hidden_size
    WHY 4×: empirically optimal ratio; enough capacity for feature transformation.
    WHY GELU: smoother than ReLU; outperforms ReLU on NLP tasks.
    """
    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, hidden_size * 4)
        self.linear2 = nn.Linear(hidden_size * 4, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))


class EncoderLayer(nn.Module):
    """Single transformer encoder layer with two sublayers + residual connections."""
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(hidden_size, num_heads, dropout)
        self.ffn       = FeedForward(hidden_size, dropout)
        self.norm1     = nn.LayerNorm(hidden_size)
        self.norm2     = nn.LayerNorm(hidden_size)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # ── Sublayer 1: Self-Attention + Add & Norm ──────────────────────────
        # query = key = value = x  →  self-attention
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))   # residual + LayerNorm

        # ── Sublayer 2: Feed-Forward + Add & Norm ───────────────────────────
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))    # residual + LayerNorm

        return x


class TransformerEncoder(nn.Module):
    """Full encoder: embeddings + N stacked encoder layers."""
    def __init__(self, vocab_size=30522, hidden_size=256,
                 num_layers=4, num_heads=8, dropout=0.1):
        super().__init__()
        self.embeddings = TransformerEmbeddings(vocab_size, hidden_size)
        self.layers = nn.ModuleList(
            [EncoderLayer(hidden_size, num_heads, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        x = self.embeddings(input_ids, token_type_ids)

        mask = None
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)  # [b, 1, 1, seq]

        for layer in self.layers:
            x = layer(x, mask)

        return self.norm(x)   # [batch, seq_len, hidden_size]


# ── Demo ──────────────────────────────────────────────────────────────────────
encoder = TransformerEncoder(vocab_size=30522, hidden_size=256,
                              num_layers=4, num_heads=8)
ids   = torch.randint(0, 30522, (2, 20))
mask  = torch.ones(2, 20, dtype=torch.long)
out   = encoder(ids, attention_mask=mask)

print("── Transformer Encoder ───────────────────────────────────────")
print(f"Input shape      : {ids.shape}")          # [2, 20]
print(f"Output shape     : {out.shape}")          # [2, 20, 256]
print(f"[CLS] vector     : {out[:, 0, :].shape}  ← used for classification")

params = sum(p.numel() for p in encoder.parameters())
print(f"\nTotal params (small config) : {params:,}")
print("BERT-base params (real)     : 110,000,000")


---
## §6 — Transformer Decoder (GPT-style)

### What is it?
The decoder generates output one token at a time. It has **three sublayers** per layer:
1. **Masked self-attention** — each token can ONLY attend to itself and PAST tokens (causal/future masking)
2. **Cross-attention** — decoder queries attend to encoder's key/value pairs (only in encoder-decoder models)
3. **Feed-forward network** — same as encoder

### Why causal masking?
During training: if the decoder could see future tokens, it would "cheat" (look at the answer).  
At inference: the model generates token t+1 given only tokens 0..t — so the mask must match.

### The causal mask
A lower-triangular matrix of 1s. Position (i,j) = 1 only if j ≤ i.  
Positions with mask=0 get filled with -∞ before softmax → 0 attention weight.

### Decoder-only (GPT) vs Encoder-Decoder (T5/BART)
| | Decoder-only | Encoder-Decoder |
|---|---|---|
| Architecture | Decoder stack only | Full encoder + decoder |
| Cross-attention | No | Yes |
| Best for | Open-ended generation | Structured generation (translation, summarization) |
| Examples | GPT-2/3/4, Claude | T5, BART, mT5 |


In [ ]:
# ── 6a: Causal Mask ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

def make_causal_mask(seq_len, device=torch.device('cpu')):
    """
    Lower-triangular boolean mask.
    mask[i,j] = 1 if token j is visible from position i (j <= i)
              = 0 if token j is in the future (j > i) → masked to -∞
    """
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]

seq_len = 8
mask = make_causal_mask(seq_len)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(mask[0, 0].numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_title("Causal (lower-triangular) mask\n1 = can attend, 0 = blocked (-∞)")
ax.set_xlabel("Key position (j)")
ax.set_ylabel("Query position (i)")
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, int(mask[0,0,i,j].item()), ha='center', va='center',
                fontsize=9, color='white' if mask[0,0,i,j]==1 else 'gray')
plt.tight_layout()
plt.savefig('/home/claude/causal_mask.png', dpi=120, bbox_inches='tight')
plt.show()

print("Row i = what token i can see")
print("Row 0 (first token) → can only attend to itself")
print("Row 7 (last token)  → can attend to all 8 tokens")


In [ ]:
# ── 6b: Decoder Layer ────────────────────────────────────────────────────────
class DecoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        # 1. Masked self-attention (causal — only past context)
        self.masked_self_attn = MultiHeadAttention(hidden_size, num_heads, dropout)
        # 2. Cross-attention (queries from decoder, K/V from encoder output)
        self.cross_attn = MultiHeadAttention(hidden_size, num_heads, dropout)
        # 3. Position-wise FFN
        self.ffn = FeedForward(hidden_size, dropout)

        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.norm3 = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_out=None, src_mask=None, tgt_mask=None):
        # ── Sublayer 1: Masked self-attention ───────────────────────────────
        a1, _ = self.masked_self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(a1))

        # ── Sublayer 2: Cross-attention (skip for decoder-only GPT models) ──
        if encoder_out is not None:
            # Decoder queries look at encoder's keys and values
            a2, _ = self.cross_attn(x, encoder_out, encoder_out, src_mask)
            x = self.norm2(x + self.dropout(a2))

        # ── Sublayer 3: Feed-forward ─────────────────────────────────────────
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x


# ── Demo: auto-regressive generation step ────────────────────────────────────
hidden_size, num_heads = 256, 8
decoder_layer = DecoderLayer(hidden_size, num_heads)

# Encoder output (e.g. encoded French sentence)
encoder_out = torch.randn(1, 10, hidden_size)   # [batch, src_len, hidden]
# Partial target sequence generated so far
tgt = torch.randn(1, 5, hidden_size)             # [batch, tgt_len, hidden]
causal = make_causal_mask(5)                     # prevent attending to future

out = decoder_layer(tgt, encoder_out=encoder_out, tgt_mask=causal)
print("── Decoder Layer Demo ────────────────────────────────────────")
print(f"Encoder output shape : {encoder_out.shape}")
print(f"Target input shape   : {tgt.shape}")
print(f"Causal mask shape    : {causal.shape}")
print(f"Decoder output shape : {out.shape}")
print("  → Same shape as input — each position now has encoded cross-context")


---
## §7 — Pre-training Tasks

### What are they?
Self-supervised objectives that train transformers on massive unlabeled text — **no human annotations needed**.

### The three main tasks

#### MLM — Masked Language Model (BERT)
1. Randomly select 15% of tokens
2. Replace 80% → `[MASK]`, 10% → random token, 10% → keep original
3. Model predicts the original tokens at masked positions
4. Forces bidirectional context reasoning

**Why 80/10/10?** If always using `[MASK]`, the model never sees real tokens at fine-tune time → distribution mismatch. Seeing random/original tokens keeps the model robust.

#### CLM — Causal Language Model (GPT)
- Predict token t+1 given tokens 0..t at every position
- Loss = cross-entropy across ALL positions simultaneously
- Enables left-to-right generation

#### NSP — Next Sentence Prediction (original BERT, later dropped)
- Binary classification: does sentence B follow sentence A?
- Dropped by RoBERTa — found to add noise, not signal

### Why pre-training matters
A BERT model pre-trained on BooksCorpus + Wikipedia (3.3B tokens) already encodes:
- Grammar and syntax
- World knowledge (facts about people, places, events)
- Reasoning patterns
- Semantic relationships

Fine-tuning then takes just hours on a small labeled dataset.


In [ ]:
# ── 7a: MLM Masking Strategy ─────────────────────────────────────────────────
def apply_mlm_masking(input_ids, tokenizer, mlm_probability=0.15):
    """
    BERT's 3-step masking strategy:
    Step 1: Select 15% of non-special tokens as candidates
    Step 2: Of selected candidates:
            80% → replace with [MASK]
            10% → replace with a random token
            10% → keep original (but still predict)
    Step 3: Compute loss ONLY on masked positions (labels=-100 elsewhere)
    """
    labels = input_ids.clone()

    # Probability matrix — start with mlm_probability everywhere
    prob_matrix = torch.full(labels.shape, mlm_probability)

    # Zero out special tokens (should never be masked)
    for special_id in tokenizer.all_special_ids:
        prob_matrix[labels == special_id] = 0.0

    # Sample which positions to process
    masked_indices = torch.bernoulli(prob_matrix).bool()

    # Only compute loss on masked positions (-100 = ignored by cross-entropy)
    labels[~masked_indices] = -100

    # 80% → replace with [MASK]
    to_mask = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
    input_ids[to_mask] = tokenizer.mask_token_id

    # 10% → replace with random token (of remaining candidates)
    remaining = masked_indices & ~to_mask
    to_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & remaining
    random_ids = torch.randint(len(tokenizer), labels.shape, dtype=torch.long)
    input_ids[to_random] = random_ids[to_random]

    # Remaining 10% → keep original token but still predict (builds robustness)

    return input_ids, labels


# ── Demo ──────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "The quick brown fox jumps over the lazy dog."
original_ids = tokenizer(text, return_tensors="pt")["input_ids"].clone()
masked_ids, labels = apply_mlm_masking(original_ids.clone(), tokenizer)

orig_tokens   = tokenizer.convert_ids_to_tokens(original_ids[0])
masked_tokens = tokenizer.convert_ids_to_tokens(masked_ids[0])

print("── MLM Masking Example ───────────────────────────────────────")
print(f"{'Position':<10} {'Original':<14} {'Masked':<14} {'Label'}")
print("-" * 55)
for i, (o, m, l) in enumerate(zip(orig_tokens, masked_tokens, labels[0])):
    changed = " ← MASKED" if l.item() != -100 else ""
    print(f"{i:<10} {o:<14} {m:<14} {l.item()}{changed}")


In [ ]:
# ── 7b: CLM — Causal Language Model objective ───────────────────────────────
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_tok.pad_token = gpt2_tok.eos_token
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

text = "Transformers are powerful because they"
inputs = gpt2_tok(text, return_tensors="pt")

# CLM: model predicts token t+1 given tokens 0..t
# The labels are simply the input IDs shifted left by 1
with torch.no_grad():
    outputs = gpt2_model(**inputs, labels=inputs["input_ids"])

print("── CLM (Causal Language Model) ──────────────────────────────")
print(f"Input              : {text!r}")
print(f"Loss (CLM)         : {outputs.loss.item():.4f}  (lower = model is more confident)")
print(f"Logits shape       : {outputs.logits.shape}")
print(f"  [batch=1, seq_len, vocab_size={outputs.logits.shape[-1]}]")

# Show top-5 predicted next tokens
last_logits = outputs.logits[0, -1]  # logits for the next token
top5 = torch.topk(last_logits, 5)
print(f"\nTop-5 predicted next tokens after '{text}':")
for score, idx in zip(top5.values, top5.indices):
    tok = gpt2_tok.decode([idx.item()])
    print(f"  {tok!r:<20} prob={F.softmax(last_logits, dim=-1)[idx].item():.3f}")


---
## §8 — Fine-tuning

### What is it?
Fine-tuning adds a lightweight **task head** on top of the pre-trained encoder and trains the whole system on labeled data.

### The three canonical NLP fine-tuning setups

| Task | Head | Input → Output |
|------|------|----------------|
| Sequence classification | Linear on [CLS] | sentence → class label |
| Token classification (NER) | Per-token linear | each token → entity label |
| Extractive QA | Two linear layers | passage → (start_idx, end_idx) |

### Why fine-tuning is so powerful
Starting from pre-trained BERT representations:
- Only ~3 epochs needed (vs 100+ from scratch)
- Only ~20K labeled examples needed (vs millions from scratch)
- Pre-trained weights encode syntax, semantics, world knowledge

### Key hyperparameters
- Learning rate: 2e-5 to 5e-5 (much smaller than pre-training)
- Warmup: first ~6% of steps use linear LR warmup
- Weight decay: 0.01 (L2 regularization)
- Epochs: 2-4 (more → overfitting on small datasets)


In [ ]:
# ── 8a: Sequence Classification (Sentiment) ─────────────────────────────────
from transformers import AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate, numpy as np

print("── Sequence Classification Setup ─────────────────────────────")
model_name = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)

# AutoModelForSequenceClassification adds:
# Linear(hidden_size=768, num_labels=2)  on top of [CLS] hidden state
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2  # positive / negative
)
print(f"Model              : {model_name}")
print(f"Task head          : Linear({model.config.hidden_size}, 2)")
print(f"Total parameters   : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params   : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Forward pass example
text = ["I love this movie!", "This film was terrible."]
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    logits = model(**inputs).logits

probs = F.softmax(logits, dim=-1)
labels = ["Negative", "Positive"]
print(f"\nInference example:")
for t, p in zip(text, probs):
    pred = labels[p.argmax().item()]
    print(f"  '{t}' → {pred} ({p.max().item():.2%})")


In [ ]:
# ── 8b: Token Classification — NER label alignment ──────────────────────────
# The key challenge: sub-word tokenization splits words into multiple tokens,
# but NER labels are at the WORD level. We must align them.

from transformers import AutoModelForTokenClassification

print("── NER Token Classification — Label Alignment ────────────────")

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")  # cased for NER

# Example: word-level tokens and their NER tags
words  = ["Barack", "Obama", "was", "born", "in", "Hawaii", "."]
labels = ["B-PER", "I-PER", "O",     "O",    "O",  "B-LOC",  "O"]
label2id = {"O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3,
            "I-ORG": 4, "B-LOC": 5, "I-LOC": 6}

# Tokenize (is_split_into_words=True → input is already word-tokenized)
encoding = tokenizer(words, is_split_into_words=True, return_tensors="pt")
tokens   = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
word_ids = encoding.word_ids(batch_index=0)  # maps token → word index

# Align labels: first sub-token of a word gets the label, rest get -100
aligned_labels = []
prev_word = None
for word_id in word_ids:
    if word_id is None:
        aligned_labels.append(-100)           # special token → ignore
    elif word_id != prev_word:
        aligned_labels.append(label2id[labels[word_id]])  # first sub-token
    else:
        aligned_labels.append(-100)           # continuation sub-token → ignore
    prev_word = word_id

print(f"{'Token':<14} {'Word ID':<10} {'Aligned Label'}")
print("-" * 40)
id2label = {v: k for k, v in label2id.items()}
for tok, wid, lab in zip(tokens, word_ids, aligned_labels):
    lab_str = id2label.get(lab, "IGNORE") if lab != -100 else "—"
    print(f"{tok:<14} {str(wid):<10} {lab_str}")
print("\nNote: '##' continuation tokens → label -100 (ignored in loss)")


In [ ]:
# ── 8c: Extractive Question Answering ────────────────────────────────────────
from transformers import AutoModelForQuestionAnswering

print("── Extractive QA (Span Prediction) ──────────────────────────")

model_name = "deepset/bert-base-cased-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model     = AutoModelForQuestionAnswering.from_pretrained(model_name)
qa_model.eval()

question = "Who invented the transformer architecture?"
context  = ("The transformer architecture was introduced in the 2017 paper "
            "'Attention Is All You Need' by Vaswani, Shazeer, Parmar and colleagues "
            "at Google Brain. The model replaced recurrent layers with self-attention.")

inputs = qa_tokenizer(question, context, return_tensors="pt",
                      truncation=True, max_length=384)

with torch.no_grad():
    outputs = qa_model(**inputs)

# Model outputs start_logits and end_logits for every token
start_logits = outputs.start_logits   # [1, seq_len]
end_logits   = outputs.end_logits     # [1, seq_len]

# Find the span with the highest combined start+end score
start_idx = start_logits.argmax(dim=-1).item()
end_idx   = end_logits.argmax(dim=-1).item()

# Decode the predicted answer span
answer_tokens = inputs["input_ids"][0][start_idx : end_idx + 1]
answer = qa_tokenizer.decode(answer_tokens, skip_special_tokens=True)

print(f"Question  : {question}")
print(f"Context   : {context[:80]}...")
print(f"\nPredicted span : tokens [{start_idx}:{end_idx}]")
print(f"Answer         : '{answer}'")
print(f"\nThis is EXTRACTIVE QA: the answer must be a verbatim span in the context.")


---
## §9 — Seq2Seq & Decoding Strategies

### Seq2Seq models
Full encoder + decoder architecture. The encoder deeply encodes the source; the decoder generates the target one token at a time using cross-attention to read the source.

**T5 text-to-text**: every task is framed as text → text with a task prefix:
- `"summarize: [article]"` → summary
- `"translate English to French: [text]"` → translation
- `"question: [q] context: [c]"` → answer

### Decoding strategies

| Strategy | How | Best for |
|---|---|---|
| Greedy | Always pick argmax | Fast baseline |
| Beam search | Keep top-B partial sequences | Translation, structured output |
| Top-K | Sample from K highest-prob tokens | Moderate creativity |
| Top-P (nucleus) | Sample from tokens summing to P | Creative text, dialogue |
| Temperature | Divide logits by T before softmax | Controlling confidence |

### Temperature effect
- T < 1 → sharper distribution → more deterministic
- T = 1 → unchanged distribution
- T > 1 → flatter distribution → more random/creative


In [ ]:
# ── 9a: Summarization with different decoding strategies ────────────────────
from transformers import pipeline
import matplotlib.pyplot as plt

print("── Decoding Strategies Comparison ───────────────────────────")

article = (
    "The transformer architecture, introduced in 2017 by Vaswani et al. in "
    "'Attention Is All You Need', replaced recurrent networks with self-attention. "
    "This enabled parallel training on massive datasets, leading to breakthrough "
    "models like BERT (2018) for understanding and GPT (2018) for generation. "
    "These pre-trained models can be fine-tuned on small labeled datasets to "
    "achieve state-of-the-art performance across virtually every NLP task."
)

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=-1  # CPU; change to 0 for GPU
)

strategies = {
    "Greedy (no sampling)":
        dict(num_beams=1, do_sample=False),
    "Beam search (4 beams)":
        dict(num_beams=4, early_stopping=True),
    "Nucleus sampling (top_p=0.9)":
        dict(do_sample=True, top_p=0.9, temperature=0.8, num_beams=1),
}

for name, kwargs in strategies.items():
    result = summarizer(article, max_length=60, min_length=15, **kwargs)
    print(f"\n{name}:")
    print(f"  {result[0]['summary_text']}")


In [ ]:
# ── 9b: Beam Search visualised ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Conceptual diagram of beam search (3 beams, 3 steps)
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.5, 3.5)
ax.axis('off')
ax.set_title("Beam Search — 3 beams × 3 steps\n"
             "Each step: expand all beams by top vocab candidates, keep top-3 by score",
             fontsize=11)

# Step labels
for step, label in enumerate(["Start", "Step 1", "Step 2", "Step 3 (Final)"]):
    ax.text(step, 3.3, label, ha='center', fontsize=9, color='gray')

colors = ['#EEEDFE', '#E1F5EE', '#FAEEDA']
border = ['#7F77DD', '#1D9E75', '#BA7517']
toks = [
    ["<s>"],
    ["The", "A", "It"],
    ["The model", "The paper", "A new"],
    ["The model changed", "The paper showed", "A new method"],
]
scores = [
    [0.0],
    [-0.3, -0.5, -0.7],
    [-0.6, -0.9, -1.1],
    [-1.1, -1.5, -1.9],
]

for step in range(4):
    for i, (tok, sc) in enumerate(zip(toks[step], scores[step])):
        c = colors[i] if step > 0 else '#F1EFE8'
        b = border[i] if step > 0 else '#888780'
        y = 2.5 - i * 1.0 if step > 0 else 1.5
        rect = mpatches.FancyBboxPatch((step-0.38, y-0.18), 0.76, 0.36,
                                        boxstyle="round,pad=0.05",
                                        facecolor=c, edgecolor=b, linewidth=1)
        ax.add_patch(rect)
        ax.text(step, y, tok, ha='center', va='center', fontsize=8.5, fontweight='500')
        ax.text(step, y-0.3, f"logP={sc:.1f}", ha='center', va='center',
                fontsize=7, color='gray')
        if step > 0:
            prev_y = 1.5 if step == 1 else 2.5 - i * 1.0
            ax.annotate("", xy=(step-0.38, y), xytext=(step-0.62, prev_y),
                        arrowprops=dict(arrowstyle='->', color=b, lw=0.8))

ax.text(3.5, 1.5, "Best beam
(highest
cumulative
log-prob)",
        ha='left', va='center', fontsize=8, color='#3B6D11')

plt.tight_layout()
plt.savefig('/home/claude/beam_search.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 9c: Temperature effect on token probabilities ───────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Simulate logits for 6 candidate tokens
tokens_demo = ["cat", "dog", "bird", "fish", "lion", "bear"]
logits_raw  = np.array([3.0, 2.0, 1.5, 1.0, 0.5, -0.5])

temperatures = [0.3, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, T in zip(axes, temperatures):
    scaled = logits_raw / T
    probs  = np.exp(scaled) / np.exp(scaled).sum()
    bars   = ax.bar(tokens_demo, probs, color='#CECBF6', edgecolor='#534AB7', lw=0.8)
    ax.set_ylim(0, 1)
    ax.set_title(f"Temperature T = {T}\n"
                 f"{'(confident/sharp)' if T<1 else '(uniform/creative)' if T>1 else '(baseline)'}",
                 fontsize=10)
    ax.set_ylabel("Probability")
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, prob + 0.01,
                f"{prob:.2f}", ha='center', fontsize=8)

plt.suptitle("Temperature scaling: T<1 sharpens, T>1 flattens the distribution",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/temperature.png', dpi=120, bbox_inches='tight')
plt.show()


---
## §10 — Evaluation Metrics

### BLEU (Bilingual Evaluation Understudy)
- Measures **n-gram precision** between prediction and reference
- Computes for n=1,2,3,4 and takes geometric mean
- Applies **brevity penalty** to prevent gaming with very short outputs
- Standard for **machine translation**

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)
- **ROUGE-N**: n-gram recall overlap
- **ROUGE-L**: longest common subsequence (order-aware)
- Standard for **summarization**

### seqeval F1 (Named Entity Recognition)
- Entity-level: the ENTIRE span must match (type + boundaries)
- F1 = 2 × (precision × recall) / (precision + recall)
- Standard for **NER, chunking, POS tagging**

### QA Token-level F1
- Measures token overlap between predicted and gold answer spans
- More lenient than exact match (EM)

### Limitations of automated metrics
- BLEU doesn't correlate perfectly with human judgment (~0.6)
- ROUGE can reward fluent but factually wrong summaries
- None measure coherence, factuality, or common sense
- Always supplement with human evaluation for production systems


In [ ]:
# ── 10a: BLEU Score ─────────────────────────────────────────────────────────
import evaluate

bleu = evaluate.load("sacrebleu")

predictions = [
    "The transformer model was developed at Google Brain.",
    "Attention mechanisms allow tokens to attend to each other.",
]
references = [
    "The transformer architecture was invented at Google Brain.",
    "Attention lets every token look at every other token.",
]

result = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]   # SacreBLEU: list of lists
)

print("── BLEU Score ────────────────────────────────────────────────")
print(f"BLEU score        : {result['score']:.2f}  (0–100 scale)")
print(f"n-gram precisions : {[round(p,2) for p in result['precisions']]}")
print(f"  1-gram / 2-gram / 3-gram / 4-gram precision")
print(f"Brevity penalty   : {result['bp']:.4f}  (1.0 = no penalty)")
print(f"\nInterpretation:")
print(f"  < 10 : Almost useless")
print(f"  10-20: Understandable with effort")
print(f"  20-30: Reasonable quality")
print(f"  30-40: Good translation")
print(f"  > 40 : Near human quality")


In [ ]:
# ── 10b: ROUGE Score ─────────────────────────────────────────────────────────
rouge = evaluate.load("rouge")

predictions = [
    "Transformers use self-attention to model language relationships.",
    "BERT is pre-trained using masked language modeling.",
]
references = [
    "Transformer models use self-attention mechanisms to understand language.",
    "BERT uses masked language model pre-training on large text corpora.",
]

result = rouge.compute(predictions=predictions, references=references)

print("── ROUGE Scores ──────────────────────────────────────────────")
for metric, score in result.items():
    print(f"{metric.upper():<12} : {score:.4f}")
print()
print("ROUGE-1  : unigram overlap (individual words)")
print("ROUGE-2  : bigram overlap (word pairs)")
print("ROUGE-L  : longest common subsequence (respects word order)")
print("ROUGE-Lsum: LCS applied at sentence level (for multi-sentence summaries)")


In [ ]:
# ── 10c: seqeval F1 for NER ─────────────────────────────────────────────────
seqeval = evaluate.load("seqeval")

# IOB2 format: B-XXX = beginning of entity, I-XXX = inside, O = outside
predictions = [
    ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O"],
    ["O", "B-LOC", "O", "O", "B-PER", "O"],
]
references = [
    ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O"],
    ["O", "B-LOC", "O", "O", "B-PER", "I-PER"],   # I-PER missed
]

result = seqeval.compute(predictions=predictions, references=references)

print("── seqeval NER F1 ────────────────────────────────────────────")
print(f"Overall F1        : {result['overall_f1']:.4f}")
print(f"Overall Precision : {result['overall_precision']:.4f}")
print(f"Overall Recall    : {result['overall_recall']:.4f}")
print(f"Overall Accuracy  : {result['overall_accuracy']:.4f}")
print()
if "PER" in result:
    print(f"PER F1            : {result['PER']['f1']:.4f}")
if "ORG" in result:
    print(f"ORG F1            : {result['ORG']['f1']:.4f}")
if "LOC" in result:
    print(f"LOC F1            : {result['LOC']['f1']:.4f}")
print()
print("Note: seqeval is STRICT — the entire span must match.")
print("      Predicting 'B-PER' correctly but missing 'I-PER' = WRONG entity.")


In [ ]:
# ── 10d: Summary comparison of all metrics ───────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

metrics = {
    "BLEU": {
        "Task": "Translation",
        "Measures": "N-gram precision",
        "Scale": "0–100",
        "Score (example)": 28.5,
        "Max": 100
    },
    "ROUGE-L": {
        "Task": "Summarization",
        "Measures": "LCS recall",
        "Scale": "0–1",
        "Score (example)": 0.47,
        "Max": 1
    },
    "NER F1": {
        "Task": "Named Entity Recog.",
        "Measures": "Entity span F1",
        "Scale": "0–1",
        "Score (example)": 0.83,
        "Max": 1
    },
    "QA F1": {
        "Task": "Question Answering",
        "Measures": "Token overlap F1",
        "Scale": "0–1",
        "Score (example)": 0.71,
        "Max": 1
    },
    "Accuracy": {
        "Task": "Classification",
        "Measures": "Correct / Total",
        "Scale": "0–1",
        "Score (example)": 0.93,
        "Max": 1
    },
}

fig, ax = plt.subplots(figsize=(10, 4))
names = list(metrics.keys())
scores_norm = [v["Score (example)"] / v["Max"] for v in metrics.values()]
scores_raw  = [v["Score (example)"] for v in metrics.values()]
tasks       = [v["Task"] for v in metrics.values()]

colors = ['#CECBF6', '#9FE1CB', '#F5C4B3', '#FAC775', '#B5D4F4']
bars = ax.barh(names, scores_norm, color=colors, edgecolor='white', height=0.55)

for i, (bar, raw, task) in enumerate(zip(bars, scores_raw, tasks)):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{raw} ({task})", va='center', fontsize=9)

ax.set_xlim(0, 1.5)
ax.set_xlabel("Normalised score (0 = worst, 1 = best)")
ax.set_title("NLP Evaluation Metrics — Representative Scores\n"
             "(BLEU divided by 100 for normalisation)", fontsize=11)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.set_frame_on(False)
ax.tick_params(left=False)

plt.tight_layout()
plt.savefig('/home/claude/metrics_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Summary — The Complete NLP Transformer Pipeline

```
Raw Text
   │
   ▼
§1  Tokenization       sub-word → integer IDs + attention mask
   │
   ▼
§2  Embeddings         token + positional + segment → [batch, seq, hidden]
   │
   ▼
§3  Self-Attention     Q·Kᵀ/√d_k → softmax → weighted sum of V
   │
   ▼
§4  Multi-Head Attn   H parallel attention heads → concat → project
   │
   ▼
   ├──────────────────────────────────────┐
   │                                      │
§5  Encoder (BERT)                §6  Decoder (GPT / T5)
   Bidirectional                      Causal mask + cross-attn
   [CLS] for classification           Auto-regressive generation
   │                                      │
   └──────────────┬───────────────────────┘
                  │
                  ▼
§7  Pre-training              MLM (BERT) / CLM (GPT) on billions of tokens
                  │
                  ▼
§8  Fine-tuning               Add task head, train on small labeled dataset
                  │
                  ▼
§9  Seq2Seq + Decoding         Greedy / Beam / Nucleus / Temperature
                  │
                  ▼
§10 Evaluation                 BLEU / ROUGE / F1 / seqeval
```

### Key numbers to remember
| Fact | Value |
|------|-------|
| BERT-base parameters | 110M |
| BERT hidden size | 768 |
| BERT attention heads | 12 |
| BERT max sequence length | 512 |
| Typical fine-tuning LR | 2e-5 |
| MLM masking rate | 15% |
| FFN expansion factor | 4× |
| Scaling factor in attention | √d_k |

### Further reading
- "Attention Is All You Need" — Vaswani et al. 2017
- "BERT: Pre-training of Deep Bidirectional Transformers" — Devlin et al. 2019
- "Language Models are Unsupervised Multitask Learners" (GPT-2) — Radford et al. 2019
- "Exploring the Limits of Transfer Learning with T5" — Raffel et al. 2020
- HuggingFace documentation: https://huggingface.co/docs/transformers
